In [ ]:
%pip install mistral-inference
%pip install mistral-common
%pip install --upgrade huggingface_hub

In [ ]:
# download the model ----> this step might take some time (25GB of params)
from huggingface_hub import snapshot_download
from pathlib import Path

mistral_models_path = Path.joinpath('mistral_models', 'Pixtral')
mistral_models_path.mkdir(parents=True, exist_ok=True)

snapshot_download(repo_id="mistralai/Pixtral-12B-2409", allow_patterns=["params.json", "consolidated.safetensors", "tekken.json"], local_dir=mistral_models_path)

# load the model 
from mistral_inference.transformer import Transformer
from mistral_inference.generate import generate

from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.protocol.instruct.messages import UserMessage, TextChunk, ImageChunk
from mistral_common.protocol.instruct.request import ChatCompletionRequest

# Initialize tokenizer and model
tokenizer = MistralTokenizer.from_file(f"{mistral_models_path}/tekken.json")
model = Transformer.from_folder(mistral_models_path)

### Helper func

In [ ]:
import base64

# Helper function to load and encode image
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

### Demo Sanity Check

In [ ]:
tokenizer = MistralTokenizer.from_file(f"{mistral_models_path}/tekken.json")
model = Transformer.from_folder(mistral_models_path)

# Run the model 
img_path = "data/converted/19.jpg"

img_input = encode_image(img_path)
prompt = "For this task, act as a pathologist who is studying extrachromosomal DNA (ecDNA). Consider this metaphase image containing ecDNA. Count the number of ecDNA."

completion_request = ChatCompletionRequest(messages=[UserMessage(content=[ImageChunk(image_url=img_input), TextChunk(text=prompt)])])

encoded = tokenizer.encode_chat_completion(completion_request)

images = encoded.images
tokens = encoded.tokens

out_tokens, _ = generate([tokens], model, images=[images], max_tokens=128, temperature=0.35, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)
result = tokenizer.decode(out_tokens[0])

print(result)

In [ ]:
# Run the model once, on a single input image
def zero_shot_one_img(img_path, prompt):
    
    img_input = encode_image(img_path)
    completion_request = ChatCompletionRequest(messages=[UserMessage(content=[ImageChunk(image_url=img_input), TextChunk(text=prompt)])])

    encoded = tokenizer.encode_chat_completion(completion_request)

    images = encoded.images
    tokens = encoded.tokens

    out_tokens, _ = generate([tokens], model, images=[images], max_tokens=128, temperature=0.35, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)
    result = tokenizer.decode(out_tokens[0])

    return (result)

In [ ]:
# Run the model for 3 images and one prompt
def zero_shot_one_img(img_list: list[3], prompt):
    
    img_list  = [encode_image(img) for img in img_list]
    
    # Send request for 3 images
    completion_request = ChatCompletionRequest(
        messages=[
            UserMessage(content=[
                ImageURLChunk(image_url=img_list[0]), ImageURLChunk(image_url=img_list[1]), ImageURLChunk(image_url=img_list[2]), TextChunk(text=prompt)
                ])
            ]
        )

    encoded = tokenizer.encode_chat_completion(completion_request)

    images = encoded.images
    tokens = encoded.tokens

    out_tokens, _ = generate([tokens], model, images=[images], max_tokens=128, temperature=0.35, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)
    result = tokenizer.decode(out_tokens[0])

    return (result)

In [ ]:
def generate_prediction(image, root_folder = 'test_images', system_message='', experiment='0-shot'):
    img = Image.open(img_path)
    system_message = '''

    '''
    prompt = PromptTemplate(
            template = 
            '''
                {system_message}
                How many ecDNA are in the image? Return an integer number in JSON format.
            '''
    )

    formatted_prompt = prompt.format(system_message=system_message)

    # Create a conversation history; this can include a single question from the user or chat structure before asking the main question (Chaining)
    messages = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': 'This is an image of a cell with nuclei, chromosomes, and ecDNA.'}
                ]
            },

            {
                'role': 'assistant',
                'content': 'Thank you for the information. What do you want me to do with the cell image?'
            },
            
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': formatted_prompt},
                ]
            }
        ]

    # Create text to pass into the processor
    text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, add_vision_id=True
        )

    # Create inputs to be passed into the model
    inputs = processor(
            text=[text],
            images=[img, img],
            padding=True,
            return_tensors="pt",
        )

    inputs = inputs.to(device)

    generated_ids = model.generate(
            **inputs, max_new_tokens=128,
            temperature=0.8,  
            top_k=10,         
            top_p=0.95,      
            do_sample=True ,   
            repetition_penalty=1.05
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return output_text  